In [3]:
# Data Loading

# import necessary libraries
from dotenv import load_dotenv
import os
from supabase import create_client
import pandas as pd
import plotly.express as px

# Load environment variables and create Supabase client
load_dotenv()
supabase = create_client(os.getenv("SUPABASE_URL"), os.getenv("SUPABASE_KEY"))

# Function to fetch data from Supabase in chunks
def get_data(table_name):
    data = []
    page_size = 1000
    page = 0
    while True:
        response = supabase.table(table_name).select("*").range(page * page_size, (page + 1) * page_size - 1).execute()
        data.extend(response.data)
        if len(response.data) < page_size:
            break
        page += 1
    return pd.DataFrame(data)

# Fetch data from Supabase tables
df_sales = get_data("sales")
df_customers = get_data("customers")

# Merge the sales and customers dataframes on customer_id
df=pd.merge(df_sales, df_customers[['customer_id', 'first_name', 'last_name', 'email', 'phone', 'city']], on='customer_id', how='left')
#-------------
# 1. Viitekuupäev
today = pd.Timestamp('2025-02-28')
# Muuda sale_date tavaliseks pandas datetime tüübiks
df['sale_date'] = pd.to_datetime(
    df['sale_date'].astype(str),
    errors='coerce'
)

# 2. RECENCY
# Iga kliendi viimase ostu kuupäev
recency = (
    df.groupby('customer_id')['sale_date']
      .max()
      .reset_index()
)

recency['recency_days'] = (today - recency['sale_date']).dt.days

# Jätame alles ainult vajalikud veerud
recency = recency[['customer_id', 'recency_days']]


# 3. FREQUENCY
# Iga kliendi ostude arv
frequency = (
    df.groupby('customer_id')['sale_id']
      .count()
      .reset_index(name='frequency')
)


# 4. MONETARY
# Iga kliendi kogukulutus
monetary = (
    df.groupby('customer_id')['total_price']
      .sum()
      .reset_index(name='monetary')
)


# 5. Liidame R, F ja M üheks tabeliks
rfm = pd.merge(recency, frequency, on='customer_id')
rfm = pd.merge(rfm, monetary, on='customer_id')


# RFM skoorid 1–5

# Recency:
# mida väiksem päevade arv, seda parem ehk kõrgem skoor
rfm['R_score'] = pd.qcut(
    rfm['recency_days'],
    5,
    labels=[5, 4, 3, 2, 1]
)

# Frequency:
# suurem ostude arv = kõrgem skoor
# rank(method='first') aitab korduvate väärtuste korral
rfm['F_score'] = pd.qcut(
    rfm['frequency'].rank(method='first'),
    5,
    labels=[1, 2, 3, 4, 5]
)

# Monetary:
# suurem kogukulutus = kõrgem skoor
rfm['M_score'] = pd.qcut(
    rfm['monetary'],
    5,
    labels=[1, 2, 3, 4, 5]
)


# Muudame skoorid arvudeks
rfm['R_score'] = rfm['R_score'].astype(int)
rfm['F_score'] = rfm['F_score'].astype(int)
rfm['M_score'] = rfm['M_score'].astype(int)


# 6. RFM koguskoor
rfm['RFM_Score'] = (
    rfm['R_score']
    + rfm['F_score']
    + rfm['M_score']
)
# Segmenteerimisfunktsioon
def segment_customer(score):
    if score >= 13:
        return 'VIP Champions'
    elif score >= 10:
        return 'Loyal'
    elif score >= 7:
        return 'Potential'
    elif score >= 4:
        return 'At Risk'
    else:
        return 'Lost'


# Määra igale kliendile segment
rfm['Segment'] = rfm['RFM_Score'].apply(segment_customer)
# Segmentide kokkuvõte
segment_summary = (
    rfm.groupby('Segment')
       .agg(
           klientide_arv=('customer_id', 'count')
       )
       .reset_index()
)

# Osakaal protsentides
segment_summary['osakaal_pct'] = (
    segment_summary['klientide_arv']
    / segment_summary['klientide_arv'].sum()
    * 100
).round(2)
# Roll D
# segmentide jaotus
segment_counts = (
    rfm['Segment']
    .value_counts()
    .reset_index()
)

segment_counts.columns = ['Segment', 'Klientide_arv']

fig1 = px.bar(
    segment_counts,
    x='Segment',
    y='Klientide_arv',
    title='UrbanStyle kliendisegmentide jaotus',
    labels={
        'Segment': 'Kliendisegment',
        'Klientide_arv': 'Klientide arv'
    },
    text='Klientide_arv'
)

fig1.update_layout(
    showlegend=False
)

fig1.show()

# hajuvusdiagramm
fig2 = px.scatter(
    rfm,
    x='recency_days',
    y='monetary',
    color='Segment',
    size='frequency',
    hover_data=['customer_id', 'frequency', 'RFM_Score'],
    title='UrbanStyle kliendisegmendid RFM analüüsi põhjal',
    labels={
        'recency_days': 'Päevi viimasest ostust',
        'monetary': 'Kogukulutus (€)',
        'frequency': 'Ostude arv',
        'Segment': 'Segment'
    }
)

fig2.show()

# TOP 10 VIP klienti
# Lisa kliendi nimi RFM tabelisse
customer_names = (
    df[['customer_id', 'first_name', 'last_name']]
    .drop_duplicates(subset='customer_id')
)

rfm = rfm.merge(customer_names, on='customer_id', how='left')

# Loo täisnimi
rfm['Kliendi_nimi'] = (
    rfm['first_name'].fillna('') + ' ' + rfm['last_name'].fillna('')
).str.strip()

top_vip = (
    rfm[rfm['Segment'] == 'VIP Champions']
    .nlargest(10, 'monetary')
)

fig3 = px.bar(
    top_vip,
    x='Kliendi_nimi',
    y='monetary',
    title='Top 10 VIP klienti kogukulutuse järgi',
    labels={
        'Kliendi_nimi': 'Kliendi nimi',
        'monetary': 'Kogukulutus (€)'
    },
    text_auto='.2f'
)

fig3.show()

# äritõlgendus
# ============================================
# ÄRITÕLGENDUSE NÄITAJAD
# ============================================

vip_count = (rfm['Segment'] == 'VIP Champions').sum()

vip_revenue = rfm.loc[
    rfm['Segment'] == 'VIP Champions',
    'monetary'
].sum()

total_revenue = rfm['monetary'].sum()

vip_revenue_share = vip_revenue / total_revenue * 100

at_risk_count = (rfm['Segment'] == 'At Risk').sum()

lost_count = (rfm['Segment'] == 'Lost').sum()

print("VIP kliente:", vip_count)
print("VIP klientide kogukulutus:", round(vip_revenue, 2), "€")
print("VIP klientide käibeosakaal:", round(vip_revenue_share, 2), "%")
print("At Risk kliente:", at_risk_count)
print("Lost kliente:", lost_count)

# segmentide kokkuvõttev tabel
segment_business = (
    rfm.groupby('Segment')
       .agg(
           klientide_arv=('customer_id', 'count'),
           kogukulutus=('monetary', 'sum'),
           keskmine_kulutus=('monetary', 'mean'),
           keskmine_ostude_arv=('frequency', 'mean')
       )
       .reset_index()
)

segment_business['käibe_osakaal_pct'] = (
    segment_business['kogukulutus']
    / segment_business['kogukulutus'].sum()
    * 100
).round(2)

segment_business

# äritõlgendus Markole
print(
    f"RFM analüüsi põhjal kuulub VIP Champions segmenti {vip_count} klienti, "
    f"kes moodustavad {vip_revenue_share:.1f}% analüüsitud klientide kogukulutusest. "
    f"At Risk segmenti kuulub {at_risk_count} klienti, kelle puhul on märgata "
    f"pikemat aega viimasest ostust ning seetõttu suuremat kliendi kaotamise riski. "
    f"Lost segmenti kuulub {lost_count} klienti, kelle ostuaktiivsus on kõige madalam. "
    f"Tulemused näitavad, et väärtuslike klientide hoidmise kõrval tasub eraldi "
    f"tegeleda passiivseks muutuvate klientide taasaktiveerimisega."
)

VIP kliente: 446
VIP klientide kogukulutus: 1115541.29 €
VIP klientide käibeosakaal: 42.53 %
At Risk kliente: 535
Lost kliente: 109
RFM analüüsi põhjal kuulub VIP Champions segmenti 446 klienti, kes moodustavad 42.5% analüüsitud klientide kogukulutusest. At Risk segmenti kuulub 535 klienti, kelle puhul on märgata pikemat aega viimasest ostust ning seetõttu suuremat kliendi kaotamise riski. Lost segmenti kuulub 109 klienti, kelle ostuaktiivsus on kõige madalam. Tulemused näitavad, et väärtuslike klientide hoidmise kõrval tasub eraldi tegeleda passiivseks muutuvate klientide taasaktiveerimisega.
